# Notebook 3: 加权融合心率估计

核心实验: 用窗口级分布加权融合多个简单场景参数的心率估计。

对每个波比跳文件, 分别用各简单运动 JSON 中的最优参数求解, 同时建立默认参数基线,
然后按分类器输出的分布加权融合, 与均匀平均、单参数集及默认基线对比误差。

**参数加载**: 使用 `load_report()` + `_merge()` 从 JSON 直接构建 `SolverParams`, 与 GUI 求解方式完全一致。

**P0 修复**: 多波比文件按 stem 独立处理, 禁止跨文件混叠。
**P2 修复**: 波比跳基线直接从 JSON 读取, 按 stem 匹配。

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import pickle
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

from ppg_hr.params import SolverParams
from ppg_hr.core.heart_rate_solver import solve_from_arrays, load_raw_data
from ppg_hr.visualization.result_viewer import load_report, _merge

# ---------- 路径配置 ----------
PROJECT_ROOT = Path(r"D:\data\PPG_HeartRate\Algorithm\outline-PPGtoHR")
DATA_DIR = PROJECT_ROOT / "docs" / "research" / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "docs" / "research" / "artifacts"
JSON_DIR = DATA_DIR / "json"

# ---------- 绘图配置 ----------
plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

## 1. 运动场景定义与参数配置

定义运动类别映射, 并从 JSON 文件直接加载各场景的优化参数。
使用与 GUI `result_viewer.py` 相同的 `load_report()` + `_merge()` 模式构建 `SolverParams`。

In [ ]:
# ---------- 运动场景定义 ----------
EXERCISES = {
    "jump_rope":    {"prefix": "tiaosheng", "label": "跳绳"},
    "arm_curl":     {"prefix": "wanju",     "label": "手臂弯举"},
    "push_up":      {"prefix": "fuwo",      "label": "俯卧撑"},
    "jumping_jack": {"prefix": "kaihe",     "label": "开合跳"},
    "burpee":       {"prefix": "bobi",      "label": "波比跳"},
}
PREFIX_MAP = {info["prefix"]: name for name, info in EXERCISES.items()}

# ---------- 从 JSON 文件直接加载参数 ----------
def parse_json_stem(fp):
    """从 JSON 文件名解析运动类别, 如 multi_tiaosheng2-green-lms-best_params.json"""
    m = re.match(r"^(multi_([a-z]+)\d+)-", fp.name)
    if not m:
        return None, None
    prefix = m.group(2)
    stem = m.group(1)
    return PREFIX_MAP.get(prefix), stem


# 简单运动: exercise -> {report, json_path}
exercise_reports = {}
# 波比跳: stem -> {report, json_path}
burpee_reports = {}

for json_fp in sorted(JSON_DIR.glob("*-best_params.json")):
    exercise, stem = parse_json_stem(json_fp)
    if exercise is None:
        print(f"  跳过无法解析: {json_fp.name}")
        continue
    report = load_report(json_fp)
    if exercise == "burpee":
        burpee_reports[stem] = {"report": report, "json_path": json_fp}
        print(f"  波比跳基线: {stem}, min_err_hf={report['min_err_hf']:.2f} BPM")
    else:
        exercise_reports[exercise] = {"report": report, "json_path": json_fp}
        print(f"  {EXERCISES[exercise]['label']} ({exercise}): "
              f"min_err_hf={report['min_err_hf']:.2f} BPM [{json_fp.name}]")

print(f"\n简单运动参数: {list(exercise_reports.keys())}")
print(f"波比跳基线样本: {list(burpee_reports.keys())}")

# ---------- 加载分类器输出 ----------
burpee_proba = np.load(ARTIFACTS_DIR / "burpee_window_distributions.npy")
with open(ARTIFACTS_DIR / "burpee_meta.pkl", "rb") as f:
    burpee_meta = pickle.load(f)

class_names = burpee_meta["class_names"]
print(f"\n分类器类别: {class_names}")
print(f"波比跳概率分布: {burpee_proba.shape}")

## 2. 加载波比跳原始数据

P0: 发现所有波比跳 CSV 文件, 为每个文件独立运行求解与融合。

In [3]:
# P0: 发现所有波比跳文件 (不再只取第一个)
burpee_csvs = sorted(DATA_DIR.glob("multi_bobi*.csv"))
burpee_csvs = [f for f in burpee_csvs if not f.stem.endswith("_ref")]

print("可用波比跳文件:")
for f in burpee_csvs:
    print(f"  {f.name}")

BURPEE_FILES = [(f, f.with_name(f.stem + "_ref" + f.suffix)) for f in burpee_csvs]

可用波比跳文件:
  multi_bobi1.csv
  multi_bobi2.csv


## 3. 逐文件求解与融合

P0: 对每个波比文件独立运行: 1) 各参数集求解; 2) 按 stem 筛选概率; 3) 对齐与融合。

In [ ]:
burpee_times_all = burpee_meta["times"]
burpee_files_all = burpee_meta["files"]


def compute_aae(pred_bpm, ref_bpm, mask=None):
    if mask is not None:
        pred_bpm, ref_bpm = pred_bpm[mask], ref_bpm[mask]
    return np.mean(np.abs(pred_bpm - ref_bpm))


def build_solver_params(report, csv_path, ref_path):
    """用 GUI 的 _merge() 模式构建 SolverParams (与 result_viewer.render() 一致)."""
    base = SolverParams(file_name=str(csv_path), ref_file=str(ref_path))
    strategy = str(report.get("adaptive_filter", base.adaptive_filter))
    ppg_mode = str(report.get("ppg_mode", base.ppg_mode))
    base = base.replace(adaptive_filter=strategy, ppg_mode=ppg_mode)
    return _merge(base, report.get("best_para_hf", {}))


def solve_with_json_params(csv_path, ref_path):
    """用各简单运动 JSON 参数分别求解, 返回各参数集的 HR 字典."""
    raw_data, ref_data = load_raw_data(
        SolverParams(file_name=str(csv_path), ref_file=str(ref_path))
    )
    results = {}
    for cn in class_names:
        if cn not in exercise_reports:
            continue
        p = build_solver_params(exercise_reports[cn]["report"], csv_path, ref_path)
        r = solve_from_arrays(raw_data, ref_data, p)
        results[cn] = {
            "time": r.HR[:, 0], "ref_hr_hz": r.HR[:, 1],
            "fus_hf_hz": r.HR[:, 5], "fus_acc_hz": r.HR[:, 6],
            "motion_flag": r.HR[:, 7], "err_hf": r.err_fus_hf,
        }
    # 默认参数基线
    dp = SolverParams(file_name=str(csv_path), ref_file=str(ref_path))
    dr = solve_from_arrays(raw_data, ref_data, dp)
    results["default"] = {
        "time": dr.HR[:, 0], "ref_hr_hz": dr.HR[:, 1],
        "fus_hf_hz": dr.HR[:, 5], "fus_acc_hz": dr.HR[:, 6],
        "motion_flag": dr.HR[:, 7], "err_hf": dr.err_fus_hf,
    }
    return results


print("=" * 70)
print(f"开始逐文件求解与融合 ({len(BURPEE_FILES)} 个波比文件)")
print("=" * 70)

all_fusion_results = {}

for csv_path, ref_path in BURPEE_FILES:
    stem = csv_path.stem
    print(f"\n--- {stem} ---")

    if not ref_path.is_file():
        print(f"  跳过: 缺少 ref 文件 {ref_path.name}")
        continue

    # 按 stem 筛选分类器概率
    stem_mask = burpee_files_all == stem
    proba_stem = burpee_proba[stem_mask]
    times_stem = burpee_times_all[stem_mask]
    print(f"  窗口数: {len(times_stem)}, 概率矩阵: {proba_stem.shape}")

    # 各参数集求解
    results = solve_with_json_params(csv_path, ref_path)
    for cn, r in results.items():
        print(f"  {cn}: {len(r['time'])} 窗口, AAE(HF)={r['err_hf']:.2f} BPM")

    # 波比跳基线: 直接从 JSON 读取
    baseline_aae = None
    if stem in burpee_reports:
        baseline_aae = burpee_reports[stem]["report"]["min_err_hf"]
        print(f"  直接优化基线: {baseline_aae:.2f} BPM")
    else:
        print(f"  直接优化基线: 无")

    # 时间轴对齐
    all_times = [r["time"] for r in results.values()]
    t_min = max(t[0] for t in all_times)
    t_max = min(t[-1] for t in all_times)
    t_common = np.arange(t_min, t_max + 1, 1.0)

    aligned_hr = {}
    aligned_ref = None
    aligned_motion = None
    for cn, r in results.items():
        f_interp = interp1d(r["time"], r["fus_hf_hz"], kind="linear", fill_value="extrapolate")
        aligned_hr[cn] = f_interp(t_common)
        if aligned_ref is None:
            aligned_ref = interp1d(r["time"], r["ref_hr_hz"], kind="linear", fill_value="extrapolate")(t_common)
            aligned_motion = interp1d(r["time"], r["motion_flag"], kind="nearest", fill_value="extrapolate")(t_common)

    # 概率对齐
    aligned_proba = np.zeros((len(t_common), len(class_names)))
    for i in range(len(class_names)):
        aligned_proba[:, i] = interp1d(times_stem, proba_stem[:, i],
                                        kind="linear", fill_value="extrapolate")(t_common)
    aligned_proba = np.clip(aligned_proba, 0, None)
    row_sums = aligned_proba.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    aligned_proba = aligned_proba / row_sums

    # 加权融合 (修复广播错误: hr_matrix shape (N, 4) 与 aligned_proba (N, 4) 对齐)
    hr_matrix = np.zeros((len(t_common), len(class_names)))
    for i, cn in enumerate(class_names):
        hr_matrix[:, i] = aligned_hr.get(cn, aligned_ref)
    weighted_hr_hz = np.sum(aligned_proba * hr_matrix, axis=1)
    uniform_hr_hz = np.mean(hr_matrix, axis=1)

    # BPM 转换
    ref_bpm = aligned_ref * 60
    weighted_hr_bpm = weighted_hr_hz * 60
    uniform_hr_bpm = uniform_hr_hz * 60
    default_hr_bpm = aligned_hr["default"] * 60
    motion_mask = aligned_motion > 0.5

    # 误差计算
    aae_weighted = {"all": compute_aae(weighted_hr_bpm, ref_bpm),
                    "rest": compute_aae(weighted_hr_bpm, ref_bpm, ~motion_mask),
                    "motion": compute_aae(weighted_hr_bpm, ref_bpm, motion_mask)}
    aae_uniform = {"all": compute_aae(uniform_hr_bpm, ref_bpm),
                   "rest": compute_aae(uniform_hr_bpm, ref_bpm, ~motion_mask),
                   "motion": compute_aae(uniform_hr_bpm, ref_bpm, motion_mask)}
    aae_default = {"all": compute_aae(default_hr_bpm, ref_bpm),
                   "rest": compute_aae(default_hr_bpm, ref_bpm, ~motion_mask),
                   "motion": compute_aae(default_hr_bpm, ref_bpm, motion_mask)}

    print(f"\n  AAE 结果:")
    print(f"    加权融合: All={aae_weighted['all']:.2f}, Rest={aae_weighted['rest']:.2f}, Motion={aae_weighted['motion']:.2f}")
    print(f"    均匀平均: All={aae_uniform['all']:.2f}, Rest={aae_uniform['rest']:.2f}, Motion={aae_uniform['motion']:.2f}")
    print(f"    默认参数: All={aae_default['all']:.2f}, Rest={aae_default['rest']:.2f}, Motion={aae_default['motion']:.2f}")
    if baseline_aae is not None:
        print(f"    直接优化: {baseline_aae:.2f} BPM")

    # 保存 (格式与之前兼容)
    all_fusion_results[stem] = {
        "t_common": t_common, "ref_bpm": ref_bpm,
        "weighted_hr_bpm": weighted_hr_bpm, "uniform_hr_bpm": uniform_hr_bpm,
        "default_hr_bpm": default_hr_bpm,
        "single_hr_bpm": {cn: aligned_hr[cn] * 60 for cn in class_names if cn in aligned_hr},
        "aligned_proba": aligned_proba, "motion_mask": motion_mask,
        "class_names": class_names,
        "aae_weighted": aae_weighted, "aae_uniform": aae_uniform, "aae_default": aae_default,
        "burpee_csv": str(csv_path), "burpee_stem": stem,
        "baseline_aae": baseline_aae,
    }

print("\n" + "=" * 70)
print(f"完成: {len(all_fusion_results)} 个文件的融合结果")

## 4. 可视化

逐文件绘制融合对比图。

In [ ]:
for stem, fusion in all_fusion_results.items():
    t = fusion["t_common"]
    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
    fig.suptitle(f"波比跳: {stem}", fontsize=14)

    # 面板 1: 加权融合 vs 参考
    ax = axes[0]
    ax.plot(t, fusion["ref_bpm"], "k-", label="参考 HR", linewidth=2)
    ax.plot(t, fusion["weighted_hr_bpm"], "r-", label="加权融合", linewidth=1.5, alpha=0.8)
    ax.plot(t, fusion["default_hr_bpm"], "g-.", label="默认参数", linewidth=1, alpha=0.7)
    motion_mask = fusion["motion_mask"]
    motion_starts = np.where(np.diff(motion_mask.astype(int)) == 1)[0]
    motion_ends = np.where(np.diff(motion_mask.astype(int)) == -1)[0]
    for s, e in zip(motion_starts, motion_ends):
        ax.axvspan(t[s], t[min(e, len(t)-1)], alpha=0.1, color="orange")
    ax.set_ylabel("HR (BPM)")
    ax.set_title("加权融合 vs 参考")
    ax.legend()

    # 面板 2: 各简单运动参数集独立求解
    ax = axes[1]
    ax.plot(t, fusion["ref_bpm"], "k-", label="参考 HR", linewidth=2)
    class_names = fusion["class_names"]
    colors = plt.cm.Set2(np.linspace(0, 1, len(class_names)))
    for i, cn in enumerate(class_names):
        if cn in fusion["single_hr_bpm"]:
            ax.plot(t, fusion["single_hr_bpm"][cn], "--", color=colors[i],
                    label=cn.replace("_", " ").title(), linewidth=1, alpha=0.7)
    ax.set_ylabel("HR (BPM)")
    ax.set_title("各简单运动参数集独立求解")
    ax.legend(fontsize=8, ncol=2)

    # 面板 3: 逐窗口绝对误差
    ax = axes[2]
    ax.plot(t, np.abs(fusion["weighted_hr_bpm"] - fusion["ref_bpm"]), "r-", label="加权融合", linewidth=1)
    ax.plot(t, np.abs(fusion["uniform_hr_bpm"] - fusion["ref_bpm"]), "b--", label="均匀平均", linewidth=1, alpha=0.7)
    ax.plot(t, np.abs(fusion["default_hr_bpm"] - fusion["ref_bpm"]), "g-.", label="默认参数", linewidth=1, alpha=0.7)
    ax.set_xlabel("时间 (s)")
    ax.set_ylabel("绝对误差 (BPM)")
    ax.set_title("逐窗口绝对误差")
    ax.legend()

    plt.tight_layout()
    plt.show()

## 5. 保存结果

In [ ]:
with open(ARTIFACTS_DIR / "fusion_results.pkl", "wb") as f:
    pickle.dump(all_fusion_results, f)
print(f"已保存: {ARTIFACTS_DIR / 'fusion_results.pkl'}")
print(f"  包含 {len(all_fusion_results)} 个波比文件: {list(all_fusion_results.keys())}")